### ***Layer Normalization***

#### **Layer Normalization is used in transformer model to normalize the values within Each Token's Representation.**

#### **Layer Normalization helps keep the representation in a more stable range, which makes training more stable and effectively.**

In [15]:
import torch
import torch.nn as nn

In [16]:
### 3 Tokens with 6-Dimensional Representation
x = torch.tensor([
    [2.0, 4.0, 6.0, 8.0, 10.0, 12.0],
    [1.0, 3.0, 5.0, 7.0, 9.0, 11.0],
    [10.0, 20.0, 30.0, 40.0, 50.0, 60.0]
])

print("Input:")
print(x)

print("Shape:", x.shape)

Input:
tensor([[ 2.,  4.,  6.,  8., 10., 12.],
        [ 1.,  3.,  5.,  7.,  9., 11.],
        [10., 20., 30., 40., 50., 60.]])
Shape: torch.Size([3, 6])


In [17]:
###Calculate the Mean of Each Token Independently
mean = x.mean(dim=-1,keepdim=True) #Keepdim True means keep the dimensions of input Token
print("Mean:",mean)

Mean: tensor([[ 7.],
        [ 6.],
        [35.]])


In [18]:
##Calculate the Variance
variance = x.var(dim = -1,keepdim=True,unbiased=False)
print("Variance:")
print(variance)

Variance:
tensor([[ 11.6667],
        [ 11.6667],
        [291.6667]])


In [ ]:
##Manually Normalize

eps = 1e-5 # Small value added for numerical stability
normalized_x = (x-mean)/torch.sqrt(variance+eps)

print("Normalized Output:")
print(normalized_x)


Normalized Output:
tensor([[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4639, -0.8783, -0.2928,  0.2928,  0.8783,  1.4639]])


In [20]:
###Calculate the Mean of Each Token Independently for normalized value
normalized_mean = normalized_x.mean(dim=-1,keepdim=True) 
print("Mean:",normalized_mean)

Mean: tensor([[0.],
        [0.],
        [0.]])


In [21]:
##Calculate the Variance
normalized_variance = normalized_x.var(dim = -1,keepdim=True,unbiased=False)
print("Variance:")
print(normalized_variance)

Variance:
tensor([[1.0000],
        [1.0000],
        [1.0000]])


In [22]:
### Using Pytorch Layer Norm
layer_norm = nn.LayerNorm(6)#dimensions of each token is 6  #output = gamma*normalized(x)+Beta

output = layer_norm(x)
print(output)

tensor([[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4639, -0.8783, -0.2928,  0.2928,  0.8783,  1.4639]],
       grad_fn=<NativeLayerNormBackward0>)


In [23]:
print("Weight (gamma):")
print(layer_norm.weight)

print("Bias (beta):")
print(layer_norm.bias)

Weight (gamma):
Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)
Bias (beta):
Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True)


In [24]:
print("Manual:")
print(normalized_x)

print("\nPyTorch LayerNorm:")
print(output)

print(
    "\nSame:",
    torch.allclose(normalized_x, output)
)

Manual:
tensor([[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4639, -0.8783, -0.2928,  0.2928,  0.8783,  1.4639]])

PyTorch LayerNorm:
tensor([[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4639, -0.8783, -0.2928,  0.2928,  0.8783,  1.4639]],
       grad_fn=<NativeLayerNormBackward0>)

Same: True


In [27]:
class LayerNorm(nn.Module):

    def __init__(self,emb_dim):
        super().__init__()

        self.eps = 1e-5

        ### Trainable scale parameter (gamma)
        self.scale = nn.Parameter(torch.ones(emb_dim))

        ###Trainable shift parameter (beta)
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self,x):

        #mean
        mean = x.mean(dim=-1,keepdim=True) 

        #variance
        variance = x.var(dim=-1,keepdim=True,unbiased = False)

        #normalization
        normalized_x = (x-mean)/torch.sqrt(variance+self.eps)

        ##output
        output = self.scale*normalized_x+self.shift

        return output
        

In [32]:
x = torch.tensor([
    [2.0, 4.0, 6.0, 8.0, 10.0, 12.0],
    [1.0, 3.0, 5.0, 7.0, 9.0, 11.0],
    [10.0, 20.0, 30.0, 40.0, 50.0, 60.0]
])

mean = x.mean(dim=-1, keepdim=True)
variance = x.var(dim=-1, keepdim=True, unbiased=False)
print("Mean Before Normalization:")
print(mean)
print("Variance Before Normalization:")
print(variance)

layer_norm = LayerNorm(emb_dim=6)

output = layer_norm(x)

print("Output:")
print(output)

print("Shape:")
print(output.shape)

mean = output.mean(dim=-1, keepdim=True)
variance = output.var(dim=-1, keepdim=True, unbiased=False)
print("Mean After Normalization:")
print(mean)
print("Variance After Normalization:")
print(variance)


Mean Before Normalization:
tensor([[ 7.],
        [ 6.],
        [35.]])
Variance Before Normalization:
tensor([[ 11.6667],
        [ 11.6667],
        [291.6667]])
Output:
tensor([[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
        [-1.4639, -0.8783, -0.2928,  0.2928,  0.8783,  1.4639]],
       grad_fn=<AddBackward0>)
Shape:
torch.Size([3, 6])
Mean After Normalization:
tensor([[0.],
        [0.],
        [0.]], grad_fn=<MeanBackward1>)
Variance After Normalization:
tensor([[1.0000],
        [1.0000],
        [1.0000]], grad_fn=<VarBackward0>)
